# 02 - Classical models across information sets

The central experiment. Every classical model (OLS, Ridge, RF, ET, GBR, XGBoost) is
fitted once on every information set, and the metrics and test-set predictions are
written to `results/tables/`. Notebook 04 draws every figure and table from those
files, so nothing is refitted to redraw a chart.

The split is chronological at `config.TEST_START` (2023-05-01). With the default
`config.EXTENDED_SEARCH = False` the model settings are those reported in the paper
(see `config.py`). Run time is a few minutes on a laptop.

**XGBoost note.** XGBoost predictions change slightly across library versions and
platforms, so the XGB rows may differ a little from the stored results. All other
models reproduce the stored results exactly.

In [ ]:
# Make the project package importable, from this folder or the project root.
# In Colab: mount Drive and set ROOT = Path("/content/drive/MyDrive/NEPSE_replication") instead.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nepse").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
from nepse import config, features, experiments

config.ensure_dirs()
print(f"EXTENDED_SEARCH={config.EXTENDED_SEARCH} (False = paper settings)  "
      f"test starts {config.TEST_START.date()}")

In [ ]:
ds = features.build_dataset()

print(f"{len(ds.y)} usable days | train {int(ds.train_mask.sum())} "
      f"| test {int((~ds.train_mask).sum())}")
for name, block in ds.info_sets.items():
    print(f"  {name:<16} {block.shape[1]:>4} features")

first, total = features.pca_explained_variance(ds)
print(f"PCA: first component {first:.1%}, ten components {total:.1%} of foreign-feature variance")

## Fit every model on every information set

In [ ]:
metrics_long, predictions = experiments.run_classical_grid(ds)

## Walk-forward robustness check

Gradient boosting re-estimated at six expanding origins across the test period
(Section 3.4), to check that the ranking of information sets does not depend on
the single 2023 split.

In [ ]:
walkforward = experiments.run_walk_forward(ds)     # GBR, 6 origins (config)
print(walkforward.round(2).to_string())

In [ ]:
experiments.save_classical(metrics_long, predictions, walkforward)